# 01 Data Profiling

This notebook prepares the interim analysis dataset for the DI501 project on song popularity.

## Objectives
- Load the Spotify track, album, and artist tables
- Standardize column names
- Check key uniqueness and duplicate rows
- Inspect missing values in the selected variables
- Produce descriptive statistics for RQ1 and RQ2
- Generate report-ready summary tables
- Create distribution plots for popularity, energy, and danceability
- Compute correlations, skewness, and IQR-based outlier summaries

## Research Questions
**RQ1:** How does song popularity vary across tracks with different combinations of energy and danceability?  
**RQ2:** How accurately can song popularity be predicted using danceability, energy, speechiness, tempo, valence, and duration_ms?


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 1. Define project paths

This notebook assumes the following folder structure:

- `E220663_DI501_Interim_Report/`
  - `data/`
  - `notebooks/`
  - `reports/`
    - `figures/`

Because this notebook is stored in the `notebooks` folder, the project root is defined as the parent directory of the current working directory.


In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("Reports dir  :", REPORTS_DIR)
print("Figures dir  :", FIGURES_DIR)
print("Tables dir   :", TABLES_DIR)


## 2. Load the datasets

In [ ]:
tracks = pd.read_csv(DATA_DIR / "spotify_tracks.csv")
albums = pd.read_csv(DATA_DIR / "spotify_albums.csv")
artists = pd.read_csv(DATA_DIR / "spotify_artists.csv")

print("=== DATASET SHAPES ===")
print("tracks :", tracks.shape)
print("albums :", albums.shape)
print("artists:", artists.shape)


## 3. Standardize column names

In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize column names and remove unnamed index-like columns.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with original column names.

    Returns
    -------
    pd.DataFrame
        Copy of the dataframe with lowercase, stripped column names and
        unnamed columns removed.
    """
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    unnamed_cols = [c for c in df.columns if c.startswith("unnamed")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
    return df

tracks = clean_columns(tracks)
albums = clean_columns(albums)
artists = clean_columns(artists)

print("=== CLEANED COLUMN NAMES (tracks) ===")
print(tracks.columns.tolist())


## 4. Select variables for the project

In [ ]:
# RQ1 variables
rq1_vars = ["popularity", "energy", "danceability"]

# RQ2 variables
rq2_vars = [
    "popularity",
    "danceability",
    "energy",
    "speechiness",
    "tempo",
    "valence",
    "duration_ms"
]

# Main analysis dataframe
analysis_df = tracks[["id"] + rq2_vars].copy()

print("=== ANALYSIS DATAFRAME INFO ===")
print(analysis_df.dtypes)
print()
display(analysis_df.head(3))


## 5. Key checks and duplicate checks

In [ ]:
def check_key(df: pd.DataFrame, col: str, name: str) -> dict:
    """
    Summarize whether a candidate key column is unique and complete.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset to inspect.
    col : str
        Column name expected to behave like a key.
    name : str
        Human-readable dataset name.

    Returns
    -------
    dict
        Dictionary containing row count, missing values, unique count,
        uniqueness flag, and duplicate key count.
    """
    return {
        "dataset": name,
        "key_column": col,
        "rows": len(df),
        "missing": int(df[col].isna().sum()),
        "unique": int(df[col].nunique()),
        "is_unique": bool(df[col].is_unique),
        "duplicate_keys": int(df[col].duplicated().sum())
    }

key_results = [
    check_key(tracks, "id", "tracks"),
    check_key(albums, "track_id", "albums"),
    check_key(artists, "track_id", "artists")
]

key_df = pd.DataFrame(key_results)
display(key_df)

key_df.to_csv(TABLES_DIR / "key_checks.csv", index=False)


In [ ]:
duplicate_results = pd.DataFrame({
    "dataset": ["tracks", "albums", "artists"],
    "duplicate_rows": [
        tracks.duplicated().sum(),
        albums.duplicated().sum(),
        artists.duplicated().sum()
    ]
})

display(duplicate_results)
duplicate_results.to_csv(TABLES_DIR / "duplicate_rows.csv", index=False)


## 6. Missing-value checks for selected variables

In [ ]:
missing_df = analysis_df[rq2_vars].isna().sum().reset_index()
missing_df.columns = ["variable", "missing_count"]

display(missing_df)
missing_df.to_csv(TABLES_DIR / "selected_vars_missing_values.csv", index=False)


## 7. Descriptive statistics

In [ ]:
desc_rq1 = analysis_df[rq1_vars].describe().T
desc_rq1 = desc_rq1[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]

desc_rq2 = analysis_df[rq2_vars].describe().T
desc_rq2 = desc_rq2[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]

print("=== RQ1 DESCRIPTIVE STATISTICS ===")
display(desc_rq1)

print("=== RQ2 DESCRIPTIVE STATISTICS ===")
display(desc_rq2)

desc_rq1.to_csv(TABLES_DIR / "desc_rq1.csv")
desc_rq2.to_csv(TABLES_DIR / "desc_rq2.csv")


## 8. Report-ready summary table

This compact table is suitable for the descriptive-statistics section of the interim report.


In [ ]:
report_vars = [
    "popularity",
    "energy",
    "danceability",
    "speechiness",
    "tempo",
    "valence",
    "duration_ms"
]

final_summary = analysis_df[report_vars].agg(["mean", "std", "min", "median", "max"]).T.round(3)
display(final_summary)

final_summary.to_csv(TABLES_DIR / "step3_summary_table.csv")


## 9. Distribution plots

The following plots are created for the main variables used in RQ1:
- popularity
- energy
- danceability


In [ ]:
plot_vars = ["popularity", "energy", "danceability"]

for col in plot_vars:
    plt.figure(figsize=(6, 4))
    plt.hist(analysis_df[col].dropna(), bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{col}_histogram.png", dpi=300)
    plt.show()

for col in plot_vars:
    plt.figure(figsize=(5, 4))
    plt.boxplot(analysis_df[col].dropna())
    plt.title(f"Boxplot of {col}")
    plt.ylabel(col)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{col}_boxplot.png", dpi=300)
    plt.show()


## 10. Correlation analysis

In [ ]:
corr_df = analysis_df[rq2_vars].corr().round(3)

display(corr_df)
print("=== POPULARITY CORRELATIONS ===")
display(corr_df.loc[["popularity"]].T)

corr_df.to_csv(TABLES_DIR / "correlation_matrix.csv")


## 11. Skewness analysis

In [ ]:
skew_df = analysis_df[rq2_vars].skew(numeric_only=True).sort_values(
    key=lambda s: s.abs(), ascending=False
).round(3)

display(skew_df.to_frame(name="skewness"))
skew_df.to_csv(TABLES_DIR / "skewness_selected_vars.csv")


## 12. IQR-based outlier analysis

In [ ]:
def iqr_outlier_summary(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Compute IQR-based outlier statistics for selected numeric variables.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    cols : list[str]
        List of column names to evaluate.

    Returns
    -------
    pd.DataFrame
        Summary table with quartiles, IQR, bounds, outlier counts, and
        outlier percentages.
    """
    results = []
    for col in cols:
        series = df[col].dropna()
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((series < lower) | (series > upper)).sum()

        results.append({
            "variable": col,
            "q1": round(q1, 3),
            "q3": round(q3, 3),
            "iqr": round(iqr, 3),
            "lower_bound": round(lower, 3),
            "upper_bound": round(upper, 3),
            "outlier_count": int(outliers),
            "outlier_pct": round(outliers / len(series) * 100, 2)
        })
    return pd.DataFrame(results)

iqr_df = iqr_outlier_summary(
    analysis_df,
    ["popularity", "energy", "danceability", "speechiness", "tempo", "valence", "duration_ms"]
)

display(iqr_df)
iqr_df.to_csv(TABLES_DIR / "iqr_outliers_selected_vars.csv", index=False)


## 13. Save the main analysis dataframe

In [ ]:
analysis_df.to_csv(REPORTS_DIR / "analysis_df.csv", index=False)

print("Step 3 notebook completed.")
print("Tables saved to :", TABLES_DIR)
print("Figures saved to:", FIGURES_DIR)
